# Delaunay-Watershed for segmentation masks compression

In this notebook:
- We present how to compress segmentation masks into a multimaterial mesh. Here we apply it on nuclei segmentation.
- We show that we can recompute the mask images from the mesh.

In this example, the compressed data (115 560 bytes) is about 145 times smaller than the rebuilt mask held as 64-bit integers, as the last cell counts it, and 36 times smaller than the original 16-bit mask doubled along z; 94 % of the rebuilt voxels have the original's label.

Its outputs were made in a window (its napari cell needs one); the data are downloaded by the second step.

# 0) Install Delaunay-Watershed

Along with tools for this notebook specifically.

In a terminal, in the environment this notebook runs in (not from the notebook):

```shell
pip install delaunay-watershed csbdeep "napari[all]"
```

The next cell downloads StarDist's 3-D demonstration data (nuclei masks) into `../data`.

## 1) Download nuclei data from Stardist 

In [1]:
from glob import glob

import numpy as np
from csbdeep.utils import Path, download_and_extract_zip_file

download_and_extract_zip_file(
    url       = "https://github.com/stardist/stardist/releases/download/0.3.0/demo3D.zip",
    targetdir = "../data",
    verbose   = 1)


X = sorted(glob("../data/train/images/*.tif"))
Y = sorted(glob("../data/train/masks/*.tif"))
assert all(Path(x).name==Path(y).name for x,y in zip(X,Y, strict=False))

Files missing, downloading...

 extracting...

 done.


## 2) Compute multimaterial mesh with Delaunay Watershed

A mesh is created via a "Mesh Reconstruction Algorithm". Although many variations can be created, we provide a default Mesh Reconstruction Algorithm:

In [2]:
from dw3d import get_default_mesh_reconstruction_algorithm

dist = 3
mesh_reconstruction_algorithm = get_default_mesh_reconstruction_algorithm(min_distance=dist, print_info=True)

But instead of a mesh, you can choose to obtain a compressed version of the segmentation masks:

In [3]:
import skimage.io as io
from scipy.ndimage import zoom

dist = 2
i = 2
initial_segmented_image = zoom(io.imread(Y[i]), [2,1,1],order = 0)


compressed_segmentation_dict = mesh_reconstruction_algorithm.compress_segmentation_mask(initial_segmented_image)

Computing EDT ...


EDT computed in  0.16
Searching local extremas ...


Number of local minimas : 3325
Number of local maxes : 342


Delaunay Tesselation built in 0.072 seconds


Graph build in  0.225
Watershed done in 0.094 seconds.


Computing EDT ...
EDT computed in  0.17
Searching local extremas ...


Number of local minimas : 3475
Number of local maxes : 335
Delaunay Tesselation built in 0.076 seconds


Graph build in  0.198
Watershed done in 0.126 seconds.


.../site-packages/dw3d/reconstruction_algorithm.py:1227: EnclosedMediumWarning: 16 enclosed pockets of medium left to the cells around them (a pocket becomes its own region when its inscribed radius is at least 4 voxels and two or more cells line it): 4 voxels (inscribed radius 1) left to cell 35 (lined by cells 35, 95): below the size bound; 4 voxels (inscribed radius 1) left to cell 28 (lined by cells 28, 96): below the size bound; 2 voxels (inscribed radius 1) left to cell 44 (lined by cells 44, 74): below the size bound; 2 voxels (inscribed radius 1) left to cell 44 (lined by cells 44, 74, 81): below the size bound; 2 voxels (inscribed radius 1) left to the exterior medium (lined by cells 73, 74): below the size bound; and 11 more. Give a pocket its own label in the mask if it is a lumen.
  self.construct_mesh_from_segmentation_mask(segmented_image)


Junction vertices relocated in 0.78 seconds: 179 of 219 moves accepted, 6 refused by the self-intersection guard
Thin medium margins: the mask was padded with medium on {'axis0_low': 4, 'axis0_high': 2, 'axis1_low': 1, 'axis1_high': 1, 'axis2_low': 3} (voxels)


.../site-packages/dw3d/reconstruction_algorithm.py:1227: CellVolumeDisagreementWarning: 6 of 98 cells have a mesh volume that disagrees with their voxel count (outside [0.5, 1.5] times it, non-positive, or absent from the mesh): labels 4, 17, 54, 56, 71, 92. Their interfaces are missing or misplaced and a pressure inferred for them is not a measurement; see the algorithm's `cell_volume_report`.
  self.construct_mesh_from_segmentation_mask(segmented_image)


In [4]:
print(compressed_segmentation_dict)

# Note that the last compressed segmentation dict can also be obtained via
compressed_segmentation_dict = mesh_reconstruction_algorithm.last_compressed_segmentation

{'points': array([[45., 82., 54.],
       [50., 77., 30.],
       [70., 55., 43.],
       ...,
       [16., 96., 37.],
       [42., 28., 94.],
       [15., 32., 80.]], shape=(1451, 3)), 'triangles': array([[   0,   45,  269],
       [   0,  795,   45],
       [   0,  269,  795],
       ...,
       [1359, 1394, 1363],
       [1386, 1438, 1422],
       [1442, 1447, 1444]], shape=(3264, 3), dtype=uint64), 'seeds': array([[ 14,  74,  15],
       [ 60,  43,  43],
       [107,  47,  66],
       [ 64,  87,  44],
       [ 85,  43,  20],
       [ 22, 109, 108],
       [ 41,  67,  20],
       [ 17,  64,  79],
       [ 63,  61,  86],
       [103, 105,  45],
       [ 85,  59, 103],
       [ 36, 103,  23],
       [ 63,  37, 106],
       [101,  86, 105],
       [ 82,  37,  79],
       [ 47,  64,  42],
       [ 44,  24, 101],
       [ 38,  86,  39],
       [ 88, 103,  59],
       [ 23,  39,  18],
       [ 16,  66, 103],
       [102,  21,  42],
       [ 45, 102,  79],
       [ 41,  39,  27],
       [ 

In [5]:
# It can also be saved on disk:
from dw3d import save_compressed_segmentation

save_compressed_segmentation("dict_mask.npy", compressed_segmentation_dict)

## 3) Reconstruct segmentation masks from the mesh. Visualize in napari for comparison

The napari cell below needs napari installed separately (`pip install napari`): the `viewing` extra of Delaunay-Watershed does not install it.

In [6]:
from dw3d import reconstruct_mask_from_dict, reconstruct_mask_from_saved_file_dict

segmented_image_reconstructed = reconstruct_mask_from_saved_file_dict("dict_mask.npy")
# or if you still have the dict directly
# segmented_image_reconstructed = reconstruct_mask_from_dict(compressed_segmentation_dict)

In [7]:
import napari

v = napari.Viewer()
v.add_image(segmented_image_reconstructed)
v.add_image(initial_segmented_image)

<Image layer 'initial_segmented_image' at 0x12d54a490>

## 4) Compare files size: mesh VS masks

In [8]:

npdict = np.load("dict_mask.npy",allow_pickle=True).item()
dict_memory_footprint = npdict["points"].nbytes + npdict["triangles"].nbytes + npdict["seeds"].nbytes + np.array(npdict["image_shape"]).nbytes
print("Size of Delaunay-Watershed data:",dict_memory_footprint,"bytes")
print("Size of the original segmentation image:",segmented_image_reconstructed.nbytes,"bytes")
print("Relative size:", dict_memory_footprint/segmented_image_reconstructed.nbytes)
print("Compression factor is:", segmented_image_reconstructed.nbytes/dict_memory_footprint)

Size of Delaunay-Watershed data: 115560 bytes
Size of the original segmentation image: 16777216 bytes
Relative size: 0.006887912750244141
Compression factor is: 145.18186223606784
